# Dataset Generation Comparison

This notebook compares dataset generation across backends and optimization settings for:
- the SRV task
- the unitary-compilation task

It runs the generator programmatically, measures runtime, computes the retained unique-sample ratio from saved dataset size vs requested samples, and saves a reproducible artifact bundle under `artifacts/evaluations/...`.


In [ ]:
from pathlib import Path
import time

import matplotlib.pyplot as plt
import pandas as pd
import yaml
from IPython.display import display

from notebooks.shared.bootstrap import setup_notebook_paths

PROJECT_ROOT = setup_notebook_paths()

from notebooks.shared.evaluation_artifacts import (
    make_artifact_dir,
    save_dataframe,
    save_figure,
    save_json,
    save_pickle,
    save_text,
)
from quantum_diffusion.data.dataset import DatasetGenerator
from my_genQC.utils.misc_utils import infer_torch_device


In [ ]:
# -- Edit only this cell -------------------------------------------------------

RUN_NAME = "backend_comparison"
ARTIFACT_SUBDIR = "srv_dataset_backend_comparison"
ARTIFACT_DIR = make_artifact_dir(PROJECT_ROOT, ARTIFACT_SUBDIR, RUN_NAME)

DEVICE = str(infer_torch_device())
OVERWRITE_EXISTING = False

SRV_CASES = [
    {
        "task": "srv",
        "label": "qiskit_optimized_10k",
        "gate_set": ["h", "cx"],
        "num_qubits": 5,
        "min_gates": 4,
        "max_gates": 20,
        "condition_type": "SRV",
        "backbone": "qiskit",
        "optimized": True,
        "num_samples": 10_000,
        "output_path": str(PROJECT_ROOT / "artifacts" / "datasets" / "tmp_backend_comparison" / "srv" / "qiskit_optimized_10k"),
    },
    {
        "task": "srv",
        "label": "qiskit_optimized_100k",
        "gate_set": ["h", "cx"],
        "num_qubits": 5,
        "min_gates": 4,
        "max_gates": 20,
        "condition_type": "SRV",
        "backbone": "qiskit",
        "optimized": True,
        "num_samples": 100_000,
        "output_path": str(PROJECT_ROOT / "artifacts" / "datasets" / "tmp_backend_comparison" / "srv" / "qiskit_optimized_100k"),
    },
    {
        "task": "srv",
        "label": "qiskit_not_optimized_10k",
        "gate_set": ["h", "cx"],
        "num_qubits": 5,
        "min_gates": 4,
        "max_gates": 20,
        "condition_type": "SRV",
        "backbone": "qiskit",
        "optimized": False,
        "num_samples": 10_000,
        "output_path": str(PROJECT_ROOT / "artifacts" / "datasets" / "tmp_backend_comparison" / "srv" / "qiskit_not_optimized_10k"),
    },
    {
        "task": "srv",
        "label": "qiskit_not_optimized_100k",
        "gate_set": ["h", "cx"],
        "num_qubits": 5,
        "min_gates": 4,
        "max_gates": 20,
        "condition_type": "SRV",
        "backbone": "qiskit",
        "optimized": False,
        "num_samples": 100_000,
        "output_path": str(PROJECT_ROOT / "artifacts" / "datasets" / "tmp_backend_comparison" / "srv" / "qiskit_not_optimized_100k"),
    },
    {
        "task": "srv",
        "label": "quditkit_optimized_10k",
        "gate_set": ["h", "cx"],
        "num_qubits": 5,
        "min_gates": 4,
        "max_gates": 20,
        "condition_type": "SRV",
        "backbone": "quditkit",
        "optimized": True,
        "num_samples": 10_000,
        "output_path": str(PROJECT_ROOT / "artifacts" / "datasets" / "tmp_backend_comparison" / "srv" / "quditkit_optimized_10k"),
    },
    {
        "task": "srv",
        "label": "quditkit_optimized_100k",
        "gate_set": ["h", "cx"],
        "num_qubits": 5,
        "min_gates": 4,
        "max_gates": 20,
        "condition_type": "SRV",
        "backbone": "quditkit",
        "optimized": True,
        "num_samples": 100_000,
        "output_path": str(PROJECT_ROOT / "artifacts" / "datasets" / "tmp_backend_comparison" / "srv" / "quditkit_optimized_100k"),
    },
    {
        "task": "srv",
        "label": "quditkit_not_optimized_10k",
        "gate_set": ["h", "cx"],
        "num_qubits": 5,
        "min_gates": 4,
        "max_gates": 20,
        "condition_type": "SRV",
        "backbone": "quditkit",
        "optimized": False,
        "num_samples": 10_000,
        "output_path": str(PROJECT_ROOT / "artifacts" / "datasets" / "tmp_backend_comparison" / "srv" / "quditkit_not_optimized_10k"),
    },
    {
        "task": "srv",
        "label": "quditkit_not_optimized_100k",
        "gate_set": ["h", "cx"],
        "num_qubits": 5,
        "min_gates": 4,
        "max_gates": 20,
        "condition_type": "SRV",
        "backbone": "quditkit",
        "optimized": False,
        "num_samples": 100_000,
        "output_path": str(PROJECT_ROOT / "artifacts" / "datasets" / "tmp_backend_comparison" / "srv" / "quditkit_not_optimized_100k"),
    },
]

UNITARY_CASES = [
    {
        "task": "unitary",
        "label": "qiskit_optimized_10k",
        "gate_set": ["h", "cx", "z", "x", "ccx", "swap"],
        "num_qubits": 3,
        "min_gates": 2,
        "max_gates": 12,
        "condition_type": "UNITARY",
        "backbone": "qiskit",
        "optimized": True,
        "num_samples": 10_000,
        "output_path": str(PROJECT_ROOT / "artifacts" / "datasets" / "tmp_backend_comparison" / "unitary" / "qiskit_optimized_10k"),
    },
    {
        "task": "unitary",
        "label": "qiskit_optimized_100k",
        "gate_set": ["h", "cx", "z", "x", "ccx", "swap"],
        "num_qubits": 3,
        "min_gates": 2,
        "max_gates": 12,
        "condition_type": "UNITARY",
        "backbone": "qiskit",
        "optimized": True,
        "num_samples": 100_000,
        "output_path": str(PROJECT_ROOT / "artifacts" / "datasets" / "tmp_backend_comparison" / "unitary" / "qiskit_optimized_100k"),
    },
    {
        "task": "unitary",
        "label": "qiskit_not_optimized_10k",
        "gate_set": ["h", "cx", "z", "x", "ccx", "swap"],
        "num_qubits": 3,
        "min_gates": 2,
        "max_gates": 12,
        "condition_type": "UNITARY",
        "backbone": "qiskit",
        "optimized": False,
        "num_samples": 10_000,
        "output_path": str(PROJECT_ROOT / "artifacts" / "datasets" / "tmp_backend_comparison" / "unitary" / "qiskit_not_optimized_10k"),
    },
    {
        "task": "unitary",
        "label": "qiskit_not_optimized_100k",
        "gate_set": ["h", "cx", "z", "x", "ccx", "swap"],
        "num_qubits": 3,
        "min_gates": 2,
        "max_gates": 12,
        "condition_type": "UNITARY",
        "backbone": "qiskit",
        "optimized": False,
        "num_samples": 100_000,
        "output_path": str(PROJECT_ROOT / "artifacts" / "datasets" / "tmp_backend_comparison" / "unitary" / "qiskit_not_optimized_100k"),
    },
    {
        "task": "unitary",
        "label": "quditkit_optimized_10k",
        "gate_set": ["h", "cx", "z", "x", "swap"],
        "num_qubits": 3,
        "min_gates": 2,
        "max_gates": 12,
        "condition_type": "UNITARY",
        "backbone": "quditkit",
        "optimized": True,
        "num_samples": 10_000,
        "output_path": str(PROJECT_ROOT / "artifacts" / "datasets" / "tmp_backend_comparison" / "unitary" / "quditkit_optimized_10k"),
    },
    {
        "task": "unitary",
        "label": "quditkit_optimized_100k",
        "gate_set": ["h", "cx", "z", "x", "swap"],
        "num_qubits": 3,
        "min_gates": 2,
        "max_gates": 12,
        "condition_type": "UNITARY",
        "backbone": "quditkit",
        "optimized": True,
        "num_samples": 100_000,
        "output_path": str(PROJECT_ROOT / "artifacts" / "datasets" / "tmp_backend_comparison" / "unitary" / "quditkit_optimized_100k"),
    },
    {
        "task": "unitary",
        "label": "quditkit_not_optimized_10k",
        "gate_set": ["h", "cx", "z", "x", "swap"],
        "num_qubits": 3,
        "min_gates": 2,
        "max_gates": 12,
        "condition_type": "UNITARY",
        "backbone": "quditkit",
        "optimized": False,
        "num_samples": 10_000,
        "output_path": str(PROJECT_ROOT / "artifacts" / "datasets" / "tmp_backend_comparison" / "unitary" / "quditkit_not_optimized_10k"),
    },
    {
        "task": "unitary",
        "label": "quditkit_not_optimized_100k",
        "gate_set": ["h", "cx", "z", "x", "swap"],
        "num_qubits": 3,
        "min_gates": 2,
        "max_gates": 12,
        "condition_type": "UNITARY",
        "backbone": "quditkit",
        "optimized": False,
        "num_samples": 100_000,
        "output_path": str(PROJECT_ROOT / "artifacts" / "datasets" / "tmp_backend_comparison" / "unitary" / "quditkit_not_optimized_100k"),
    },
]

CASES = SRV_CASES + UNITARY_CASES
print(f"Artifact dir: {ARTIFACT_DIR}")
print(f"Device: {DEVICE}")
print(f"Cases: {len(CASES)}")


In [ ]:
def resolve_saved_dataset_dir(case, generation_results):
    condition_key = str(case["condition_type"]).lower()
    output_path = Path(generation_results[condition_key]["output_path"])
    if (output_path / "config.yaml").exists():
        return output_path
    raise FileNotFoundError(f"Expected saved dataset under {output_path}")


def read_dataset_config(dataset_dir):
    with (dataset_dir / "config.yaml").open("r", encoding="utf-8") as handle:
        return yaml.safe_load(handle)


def summarize_case(case, generation_results, elapsed_seconds):
    dataset_dir = resolve_saved_dataset_dir(case, generation_results)
    dataset_cfg = read_dataset_config(dataset_dir)
    params = dataset_cfg.get("params", {})
    unique_samples = int(generation_results[str(case["condition_type"]).lower()]["num_samples"])
    requested_samples = int(case["num_samples"])
    unique_ratio = unique_samples / requested_samples if requested_samples else float("nan")

    return {
        "task": case["task"],
        "label": case["label"],
        "backbone": case["backbone"],
        "optimized": bool(case["optimized"]),
        "requested_samples": requested_samples,
        "unique_samples": unique_samples,
        "unique_ratio": unique_ratio,
        "elapsed_seconds": elapsed_seconds,
        "output_path": str(dataset_dir),
        "gate_set": list(case["gate_set"]),
        "num_qubits": int(case["num_qubits"]),
        "min_gates": int(case["min_gates"]),
        "max_gates": int(case["max_gates"]),
        "pad_constant": params.get("pad_constant"),
        "comment": dataset_cfg.get("comment"),
    }


def run_case(case, generator):
    output_path = Path(case["output_path"])
    if output_path.exists() and not OVERWRITE_EXISTING:
        raise FileExistsError(f"Refusing to overwrite existing output: {output_path}")

    start = time.perf_counter()
    generation_results = generator.generate_dataset(
        gate_set=case["gate_set"],
        num_qubits=case["num_qubits"],
        num_samples=case["num_samples"],
        min_gates=case["min_gates"],
        max_gates=case["max_gates"],
        backbone=case["backbone"],
        condition_type=case["condition_type"],
        optimized=case["optimized"],
        output_path=case["output_path"],
        n_jobs=4,
    )
    elapsed = time.perf_counter() - start
    return summarize_case(case, generation_results, elapsed)


generator = DatasetGenerator(device=DEVICE)
rows = []
for case in CASES:
    print(f"Running {case['task']} | {case['label']}")
    rows.append(run_case(case, generator))

results_df = pd.DataFrame(rows).sort_values(["task", "backbone", "optimized", "requested_samples"]).reset_index(drop=True)
summary_tables = {
    task: df.drop(columns=["output_path", "gate_set", "comment"]).copy()
    for task, df in results_df.groupby("task", sort=False)
}

save_dataframe(results_df, ARTIFACT_DIR / "all_runs.csv", index=False)
for task, df in summary_tables.items():
    save_dataframe(df, ARTIFACT_DIR / f"{task}_summary.csv", index=False)
save_pickle(rows, ARTIFACT_DIR / "results.pkl")
save_json({"cases": CASES, "device": DEVICE, "results": rows}, ARTIFACT_DIR / "run_config_and_results.json")
save_text(f"Saved dataset generation comparison artifacts to {ARTIFACT_DIR}\n", ARTIFACT_DIR / "README.txt")

for task, df in summary_tables.items():
    print(f"\n## {task.upper()} summary")
    display(df)


In [ ]:
def plot_metric(task_df, metric, ylabel, title):
    fig, ax = plt.subplots(figsize=(8, 4), dpi=160)
    for (backbone, optimized), subdf in task_df.groupby(["backbone", "optimized"], sort=False):
        subdf = subdf.sort_values("requested_samples")
        label = f"{backbone} | optimized={optimized}"
        ax.plot(subdf["requested_samples"], subdf[metric], marker="o", linewidth=1.8, label=label)
    ax.set_xscale("log")
    ax.set_xlabel("Requested samples")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    ax.legend()
    return fig

for task, df in results_df.groupby("task", sort=False):
    fig_time = plot_metric(df, "elapsed_seconds", "Seconds", f"{task.upper()} generation time")
    save_figure(fig_time, ARTIFACT_DIR / f"{task}_generation_time.png")
    plt.show()

    fig_unique = plot_metric(df, "unique_ratio", "Unique ratio", f"{task.upper()} unique-sample ratio")
    save_figure(fig_unique, ARTIFACT_DIR / f"{task}_unique_ratio.png")
    plt.show()


In [ ]:
results_df
